# Entrenar Modelo de Predicción Nutricional desde Colab

Este notebook permite entrenar el modelo de predicción nutricional llamando al endpoint del backend.

## 1. Configuración

In [ ]:
# URL del backend (cambiar según tu configuración)
BACKEND_URL = "http://localhost:8000"  # Local
# BACKEND_URL = "https://tu-dominio.com"  # Producción

print(f"Backend URL: {BACKEND_URL}")

## 2. Instalar dependencias

In [ ]:
!pip install requests -q

## 3. Entrenar el modelo

In [ ]:
import requests
import json
import time

# Parámetros de entrenamiento
payload = {
    "min_measurements": 2,
    "lookback_months": 24,
    "include_synthetic": True
}

endpoint = f"{BACKEND_URL.rstrip('/')}/api/v1/ml/train_model"

print(f"🚀 Iniciando entrenamiento...")
print(f"   Endpoint: {endpoint}")
print(f"   Parámetros: {json.dumps(payload, indent=2)}")
print()

try:
    start_time = time.time()
    response = requests.post(endpoint, json=payload, timeout=900)
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        result = response.json()
        
        print("✅ ENTRENAMIENTO COMPLETADO")
        print(f"   Status: {result.get('status')}")
        print(f"   Mensaje: {result.get('message')}")
        print(f"   Accuracy: {result.get('accuracy', 'N/A')}")
        print(f"   F1-Score: {result.get('f1_score', 'N/A')}")
        print(f"   Dataset: {result.get('dataset_size', 'N/A')} registros")
        print(f"   Tiempo: {result.get('training_time_seconds', elapsed):.1f}s")
        print(f"   Modelo: {result.get('model_path')}")
    else:
        print(f"❌ Error HTTP {response.status_code}")
        print(f"   Response: {response.text}")
        
except requests.exceptions.Timeout:
    print("❌ Timeout: El entrenamiento tardó más de 15 minutos")
except requests.exceptions.ConnectionError:
    print(f"❌ No se puede conectar a {BACKEND_URL}")
    print("   Verifica que el backend esté corriendo")
except Exception as e:
    print(f"❌ Error: {str(e)}")

## 4. Verificar que el modelo se cargó correctamente

In [ ]:
# Verificar health del servicio ML
health_endpoint = f"{BACKEND_URL.rstrip('/')}/api/v1/ml/health"

try:
    response = requests.get(health_endpoint, timeout=10)
    if response.status_code == 200:
        health = response.json()
        print("✅ Servicio ML disponible")
        print(json.dumps(health, indent=2))
    else:
        print(f"❌ Error: {response.status_code}")
except Exception as e:
    print(f"❌ Error: {str(e)}")